# Classic CNN (ResNet) - Model, Make Prediction

### Imports

In [1]:
from collections import defaultdict
from torch.utils.data import Subset
import kagglehub
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import os
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data import random_split
from torchvision import datasets
from torchvision import transforms
from tqdm import tqdm
import re
import random

### VMMRdb Download

In [2]:
path = kagglehub.dataset_download("prabashwara/vmmrdb-dataset")

print("Path to dataset files:", path)

100%|██████████| 11.5G/11.5G [02:18<00:00, 89.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/prabashwara/vmmrdb-dataset/versions/1


### Custom Pytorch Dataset Definition

In [7]:
class VMMRDB_Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        # Unique classes
        unique_classes = set()
        for class_name in sorted(os.listdir(root_dir)):
          class_path = os.path.join(root_dir, class_name)
          if os.path.isdir(class_path):
            stripped = re.sub(r'_\d{4}$', '', class_name)
            unique_classes.add(stripped)

        # Indicies to unique class names
        for idx, class_name in enumerate(sorted(unique_classes)):
          self.class_to_idx[class_name] = idx

        # Images to class names
        for class_name in sorted(os.listdir(root_dir)):
          class_path = os.path.join(root_dir, class_name)
          if os.path.isdir(class_path):
            stripped = re.sub(r'_\d{4}$', '', class_name)
            idx = self.class_to_idx[stripped]
            for img_name in os.listdir(class_path):
              if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

### ResNet 18 Architecture

In [8]:
class BasicBlock(nn.Module):
  def __init__(self, in_channel, out_channel, stride=1):
    super(BasicBlock, self).__init__()

    # First convolution
    self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=3, padding=1, stride=stride, bias=False)
    self.batch_norm1 = nn.BatchNorm2d(out_channel)

    # Second Convolution
    self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=1, padding=1, bias=False)
    self.batch_norm2 = nn.BatchNorm2d(out_channel)

    # Skip Connection
    self.shortcut = nn.Sequential()
    if stride != 1 or in_channel != out_channel:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channel, out_channel, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channel)
      )

    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    out = self.relu(self.batch_norm1(self.conv1(x)))
    out = self.batch_norm2(self.conv2(out))
    out += self.shortcut(x)
    out = self.relu(out)
    return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()

        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2,
                               padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = []

        # First block
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels

        # Remaining blocks
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        # First convolution step
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        # ResNet Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Fully connected portion
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


def ResNet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

### Training and Testing Dataset

In [5]:
# Used to set a max number of images per class
# Also doubles as a way to create a test set

def filter_dataset(dataset, max_per_class, min_per_class):
  random.seed(0)

  # Group indicies by label
  class_indicies = defaultdict(list)
  for idx, label in enumerate(dataset.labels):
    class_indicies[label].append(idx)

  # Find kept indicies
  kept_indicies_train = []
  kept_indicies_test = []

  for label, indices in class_indicies.items():
    train_indices = []
    test_indices = []
    if min_per_class and len(indices) < min_per_class:
      continue
    if max_per_class and len(indices) >= max_per_class:
      train_indices = random.sample(indices, max_per_class)
      test_indices = list(set(indices) - set(train_indices))
    kept_indicies_train.extend(train_indices)
    kept_indicies_test.extend(test_indices)

  return Subset(dataset, kept_indicies_train), Subset(dataset, kept_indicies_test)

In [6]:
# batch size
batch_size = 48

# Defines image preprocessing
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.4381, 0.4339, 0.4273], [0.2545, 0.2524, 0.2526])
])

# Creates test and train dataset by limiting the number of images per class
dataset_VMMRDB = VMMRDB_Dataset(path, transform=transform)
dataset_train, dataset_test = filter_dataset(dataset_VMMRDB, min_per_class=200, max_per_class=500)

# If you have the compute, uncomment the three lines below and comment out the line above
# train_size = int(0.8 * len(dataset_VMMRDB))
# test_size = len(dataset_VMMRDB) - train_size
# train_dataset, test_dataset = random_split(dataset_VMMRDB, [train_size, test_size])

# Creates the train and test loaders
train_loader = DataLoader(
    dataset_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
test_loader = DataLoader(
    dataset_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

### Training Step

In [10]:
# Defines CNN, loss and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
net = ResNet18(num_classes=len(dataset_VMMRDB.class_to_idx)).to(device)
loss_criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9)

# Number of iterations over dataset
epochs = 20

checkpoint_path = "./checkpoint.pth"
start_epoch = 0

# Trying to load trained weights
try:
    checkpoint = torch.load(checkpoint_path)
    net.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}")
except:
    print("No checkpoint found, starting fresh.")

# Training loop
curr_epoch=0
curr_loss=-1
for epoch in range(epochs):
  loop = tqdm(train_loader, leave=True)
  curr_epoch=epoch
  for X, y in loop:
    X = X.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    output = net(X)
    loss = loss_criterion(output, y)
    curr_loss=loss.item()
    loss.backward()
    optimizer.step()
    loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
    loop.set_postfix(loss=loss.item())
  print(f"Epoch {epoch+1} finished. Loss: {loss.item()}")
checkpoint = {
    "epoch": curr_epoch,
    "model_state": net.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "loss": curr_loss
}
torch.save(checkpoint, checkpoint_path)

### Load Model and Evaluating

In [12]:
# Loading model
device = "cuda" if torch.cuda.is_available() else "cpu"
net = ResNet18(num_classes=len(dataset_VMMRDB.class_to_idx)).to(device)
try:
  checkpoint_path = "./checkpoint.pth"
  checkpoint = torch.load(checkpoint_path)
  net.load_state_dict(checkpoint["model_state"])
except:
    print("No checkpoint found")

correct = 0
total = 0

# Test loop
net.eval()
with torch.no_grad():
    loop = tqdm(test_loader, leave=True)
    for X, y in loop:
        X = X.to(device)
        y = y.to(device)

        outputs = net(X)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == y).sum().item()
        total += y.size(0)
        acc = correct / total

        loop.set_description("Evaluating")
        loop.set_postfix(acc=acc)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")